In [1]:
import numpy as np
from matplotlib import pyplot as plt
from scipy.stats import gamma as gam
import sys
sys.path.append("..")
from src.data.sir_model import SIRModel
from sklearn.decomposition import PCA

from torch.utils.data import DataLoader, random_split
from functools import partial
from src.models import MarginalDensityFlow
import lightning as L
import torch
from lightning.pytorch.callbacks.early_stopping import EarlyStopping
from src.criticism import c2st
from sklearn.neighbors import KernelDensity

In [10]:
N = 10000
a = int(N * 0.8)
b = N - a
# want to increase T eventually
dataset = SIRModel(beta=0.3, gamma=0.1, prior_scale=[0.2, 0.2], N=100, T=52, observed_seed=9, n_sample=N, partial_obs=True,
                  mode="criticism")
train_data, val_data = random_split(dataset, [a, b])
train_loader = DataLoader(train_data, batch_size=200)
val_loader = DataLoader(val_data, batch_size=b)
observed_data = dataset.get_observed_data()

In [65]:
X = dataset.data.squeeze(1).numpy()
kde = KernelDensity(kernel="tophat", bandwidth="silverman").fit(X)
kde.bandwidth_

0.8098170141358446

In [66]:
mll = kde.score_samples(observed_data[0].numpy())
(kde.score_samples(X[:1000]) <= mll).mean()

np.float64(0.219)